In [37]:
import pandas
import sys
import os
sys.path.append("../")
from main import *

In [38]:
datasets = {
    "Mushrooms": get_mushrooms_data(),
    "Chess": get_chess_data(),
    "StudentSuccess": get_student_success_data(),
    "connect4": get_connect4_data(),
    "HealthyAging": get_healthy_aging_data(),
    "Car": get_car_data(),
    "Soybean": get_soybean_data(),
    "Dermatology": get_dermatology_data(),
    "Depression": get_depression_data(),
    "Vancomycin": get_vancomycin_data(),
    "MIMIC-IV": get_mimic_iv_data(),
    "Cardio": get_cardio_data(),
}

In [39]:
import os
import re

import numpy as np
import pandas as pd

RESULTS_DIR = "results"

# Matches conditions like: cholesterol = '3'
CONDITION_RE = re.compile(r"([^=,]+?)\s*=\s*'([^']*)'")



In [40]:

def normalize(name):
    """Strip surrounding whitespace (tabs included) and unify inner whitespace."""
    return re.sub(r"\s+", "_", str(name).strip())


def rule_mask(df, description):
    """Return a boolean array marking the instances covered by a rule.

    `df` must already have normalized column names.
    """
    mask = np.ones(len(df), dtype=bool)
    for attribute, value in CONDITION_RE.findall(description):
        # Compare as strings, since the csv stores values as text
        mask &= (df[normalize(attribute)].astype(str) == value).to_numpy()
    return mask

def jaccard_matrix(masks):
    """Pairwise Jaccard index between the rows of a boolean matrix."""
    m = masks.astype(np.float32)
    intersection = m @ m.T
    sizes = m.sum(axis=1)
    union = sizes[:, None] + sizes[None, :] - intersection
    # Two rules covering no instances get a Jaccard index of 0
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)



In [41]:

jaccard_results = {}  # csv name -> DataFrame with the pairwise Jaccard indexes
mean_jaccard_df = pd.DataFrame(columns=["dataset", "algorithm", "mean_jaccard"])

for csv_name in sorted(os.listdir(RESULTS_DIR)):
    if not csv_name.endswith(".csv"):
        continue

    # Expected name: [algorithm]_[dataset].csv
    algorithm, dataset = os.path.splitext(csv_name)[0].split("_", 1)
    if dataset not in datasets:
        print(f"Warning: dataset '{dataset}' not found in `datasets`, skipping {csv_name}")
        continue

    rules = pd.read_csv(os.path.join(RESULTS_DIR, csv_name), index_col=0)
    # Normalize column names so they match the attributes used in the rules
    df = datasets[dataset].rename(columns=normalize)

    # One row per rule, one column per instance
    masks = np.vstack([rule_mask(df, d) for d in rules["Description"]])

    jaccard_results[csv_name] = pd.DataFrame(
        jaccard_matrix(masks), index=rules.index, columns=rules.index
    )

    # Mean Jaccard over distinct pairs (upper triangle, without the diagonal)
    n = len(rules)
    mean_jaccard = jaccard_results[csv_name].to_numpy()[np.triu_indices(n, k=1)].mean() if n > 1 else float("nan")
    mean_jaccard_df.loc[len(mean_jaccard_df)] = [dataset, algorithm, mean_jaccard]
    # print(f"{csv_name}: {n} rules, mean pairwise Jaccard = {mean_jaccard:.4f}")

In [42]:
# Rename datasets for better readability in the final table
dataset_rename = {
    "Car" : "Car Evaluation",
    "StudentSuccess" : "Student Success",
    "connect4" : "Connect-4"
}
mean_jaccard_df["dataset"] = mean_jaccard_df["dataset"].replace(dataset_rename)

In [43]:
mean_jaccard_df = mean_jaccard_df.pivot(index="dataset", columns="algorithm", values="mean_jaccard")
mean_jaccard_df

algorithm,BSD,BerryFinder,SDMapStar
dataset,,,
Car Evaluation,0.200000,0.200000,0.200000
Cardio,0.935656,0.000000,0.935656
Chess,0.964976,0.417801,NaN
Connect-4,0.956436,0.201009,NaN
Depression,0.166124,0.166124,0.166124
Dermatology,0.214611,0.446267,NaN
MIMIC-IV,0.937845,0.000000,0.938129
Mushrooms,0.389116,0.256427,0.688916
Student Success,0.705898,0.486568,NaN


In [45]:
print(mean_jaccard_df.to_latex())

\begin{tabular}{lrrr}
\toprule
algorithm & BSD & BerryFinder & SDMapStar \\
dataset &  &  &  \\
\midrule
Car Evaluation & 0.200000 & 0.200000 & 0.200000 \\
Cardio & 0.935656 & 0.000000 & 0.935656 \\
Chess & 0.964976 & 0.417801 & NaN \\
Connect-4 & 0.956436 & 0.201009 & NaN \\
Depression & 0.166124 & 0.166124 & 0.166124 \\
Dermatology & 0.214611 & 0.446267 & NaN \\
MIMIC-IV & 0.937845 & 0.000000 & 0.938129 \\
Mushrooms & 0.389116 & 0.256427 & 0.688916 \\
Student Success & 0.705898 & 0.486568 & NaN \\
Vancomycin & 0.587242 & 0.199384 & 0.821550 \\
\bottomrule
\end{tabular}

